# Enterprise Master Data Integration Platform

**Hands-on project - Data Engineering training (DataMites)**

Consolidates customer, product and supplier master data: read the source files, clean and standardize them, validate quality, and prepare dimension-ready tables for Snowflake. The production-style version of this notebook is the AWS Glue PySpark job in this folder, with dbt models in `DBT/` and SQL checks in `SnowSQL/`.


In [1]:
# Read all CSV files
import pandas as pd

customers = pd.read_csv("../datasets/customers.csv")
products = pd.read_csv("../datasets/products.csv")
suppliers = pd.read_csv("../datasets/suppliers.csv")

# Display first 5 rows
print(customers.head())
print(products.head())
print(suppliers.head())

   customer_id    customer_name                        email     city country
0            1      James Smith      james.smith@example.com  Houston     USA
1            2     Mary Johnson     mary.johnson@example.com   Dallas     USA
2            3  Robert Williams  robert.williams@example.com   Austin     USA
3            4   Patricia Brown   patricia.brown@example.com  Chicago     USA
4            5       John Jones       john.jones@example.com  Seattle     USA
   product_id           product_name     category  unit_price
0           1         Wireless Mouse  Accessories       24.99
1           2    Mechanical Keyboard  Accessories       79.99
2           3  USB-C Docking Station  Electronics      129.99
3           4        27-inch Monitor  Electronics      229.99
4           5           Laptop Stand  Accessories       39.99
   supplier_id  ...  country
0            1  ...      USA
1            2  ...      USA
2            3  ...   Sweden
3            4  ...   Canada
4            5 

## Data quality checks
Row counts, nulls and duplicate ids in the raw files.

In [2]:
# Check shape, nulls and duplicate ids
for name, df, key in [("customers", customers, "customer_id"), ("products", products, "product_id"), ("suppliers", suppliers, "supplier_id")]:
    print(name, df.shape, "| nulls:", int(df.isnull().sum().sum()), "| duplicate ids:", int(df.duplicated(subset=[key]).sum()))

customers (50, 5) | nulls: 0 | duplicate ids: 0
products (20, 4) | nulls: 0 | duplicate ids: 0
suppliers (10, 4) | nulls: 0 | duplicate ids: 0


## Clean the master data
Trim spaces, standardize email case, drop duplicate ids - the same cleaning the Glue PySpark job (`glue_etl_job.py`) does before loading Snowflake.

In [3]:
# Clean customers
customers["customer_name"] = customers["customer_name"].str.strip()
customers["email"] = customers["email"].str.strip().str.lower()
customers["country"] = customers["country"].str.upper()
customers = customers.drop_duplicates(subset=["customer_id"])

# Clean products
products["product_name"] = products["product_name"].str.strip()
products["category"] = products["category"].str.strip()
products = products.drop_duplicates(subset=["product_id"])

# Clean suppliers
suppliers["supplier_name"] = suppliers["supplier_name"].str.strip()
suppliers["contact_email"] = suppliers["contact_email"].str.strip().str.lower()
suppliers["country"] = suppliers["country"].str.upper()
suppliers = suppliers.drop_duplicates(subset=["supplier_id"])

print("Cleaned rows -> customers:", len(customers), "| products:", len(products), "| suppliers:", len(suppliers))
print(customers.head())

Cleaned rows -> customers: 50 | products: 20 | suppliers: 10
   customer_id    customer_name                        email     city country
0            1      James Smith      james.smith@example.com  Houston     USA
1            2     Mary Johnson     mary.johnson@example.com   Dallas     USA
2            3  Robert Williams  robert.williams@example.com   Austin     USA
3            4   Patricia Brown   patricia.brown@example.com  Chicago     USA
4            5       John Jones       john.jones@example.com  Seattle     USA


## Validation after cleaning
Every check below should print 0 / False - the SQL version of these checks is in `SnowSQL/validation_checks.sql`.

In [4]:
# Validation checks (all should pass)
print("Duplicate customer ids:", int(customers.duplicated(subset=['customer_id']).sum()))
print("Null customer emails:", int(customers['email'].isnull().sum()))
print("Duplicate product ids:", int(products.duplicated(subset=['product_id']).sum()))
print("Products with non-positive price:", int((products['unit_price'] <= 0).sum()))
print("Duplicate supplier ids:", int(suppliers.duplicated(subset=['supplier_id']).sum()))

Duplicate customer ids: 0
Null customer emails: 0
Duplicate product ids: 0
Products with non-positive price: 0
Duplicate supplier ids: 0


## Master data summary
Counts by category and country - a quick view of the consolidated master data that gets loaded to Snowflake dimensions (dim_customer, dim_product, dim_supplier) through dbt.

In [5]:
# Summary views
print("Products by category:")
print(products.groupby("category")["product_id"].count().to_string())
print()
print("Customers by city (top 5):")
print(customers.groupby("city")["customer_id"].count().sort_values(ascending=False).head(5).to_string())
print()
print("Suppliers by country:")
print(suppliers.groupby("country")["supplier_id"].count().to_string())

Products by category:
category
Accessories        5
Electronics        6
Furniture          3
Networking         3
Office Supplies    1
Storage            1
Wearables          1

Customers by city (top 5):
city
Atlanta    5
Austin     5
Boston     5
Chicago    5
Dallas     5

Suppliers by country:
country
CANADA         2
GERMANY        1
SINGAPORE      1
SWEDEN         1
SWITZERLAND    1
USA            4


## Next step - AWS Glue + Snowflake + dbt
In the full pipeline, `PySpark/glue_etl_job.py` runs this same cleaning as an AWS Glue job from the S3 raw zone into Snowflake staging, dbt builds the dimension tables (`DBT/models/`), and `SnowSQL/validation_checks.sql` re-runs these quality checks inside Snowflake.